In [ ]:

# ============================================================
# STUDI KASUS WEBINAR 5
# HUMAN-COMPUTER INTERACTION & DATA VISUALIZATION
# Dataset asli: UCI Air Quality Dataset
# Seluruh implementasi dalam satu kode Google Colab
# ============================================================

# ============================================================
# SESI 1 - KONSEP HCI DAN VISUALISASI DATA
# ============================================================

from IPython.display import display, Markdown, clear_output
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from datetime import timedelta

display(Markdown("""
# STUDI KASUS: DASBOR INTERAKTIF KUALITAS UDARA

**Materi:** Human-Computer Interaction & Data Visualization

**Tujuan:** Mengembangkan prototipe dasbor yang membantu pengguna
memahami perubahan kualitas udara melalui grafik dan filter interaktif.

Dataset yang digunakan merupakan hasil pengukuran sensor kualitas
udara di Italia dari UCI Machine Learning Repository.

Prinsip yang diterapkan:
- Human-Computer Interaction (HCI)
- User-Centered Design (UCD)
- Visualisasi data
- Aksesibilitas dan usability
- Dasbor interaktif
"""))

# Mengunduh dataset asli
url = "https://archive.ics.uci.edu/static/public/360/air+quality.zip"

print("Mengunduh dataset asli dari UCI...")

try:
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as arsip:
        nama_csv = [
            nama for nama in arsip.namelist()
            if nama.lower().endswith(".csv")
            and "airqualityuci" in nama.lower()
        ]

        if not nama_csv:
            raise FileNotFoundError(
                "File CSV AirQualityUCI tidak ditemukan di dalam ZIP."
            )

        with arsip.open(nama_csv[0]) as file:
            df = pd.read_csv(
                file,
                sep=";",
                decimal=",",
                encoding="utf-8"
            )

    # Membersihkan kolom yang seluruh nilainya kosong
    df = df.dropna(axis=1, how="all")

    # Menggabungkan tanggal dan waktu
    df.columns = df.columns.str.strip()

    if "Date" not in df.columns or "Time" not in df.columns:
        raise ValueError(
            "Kolom Date atau Time tidak ditemukan. "
            "Periksa struktur dataset yang diunduh."
        )

    df["Datetime"] = pd.to_datetime(
        df["Date"].astype(str).str.strip()
        + " "
        + df["Time"].astype(str).str.strip(),
        format="%d/%m/%Y %H.%M.%S",
        errors="coerce"
    )

    df = df.dropna(subset=["Datetime"]).copy()

    # Mengubah kolom pengukuran menjadi numerik.
    # Nilai -200 dalam dataset menandakan data pengukuran hilang.
    kolom_pengukuran = [
        kolom for kolom in df.columns
        if kolom not in ["Date", "Time", "Datetime"]
    ]

    for kolom in kolom_pengukuran:
        df[kolom] = pd.to_numeric(df[kolom], errors="coerce")
        df.loc[df[kolom] == -200, kolom] = np.nan

    # Mempertahankan kolom pengukuran yang mempunyai data valid
    kolom_pengukuran = [
        kolom for kolom in kolom_pengukuran
        if df[kolom].notna().any()
    ]

    if df.empty or not kolom_pengukuran:
        raise ValueError("Dataset tidak memiliki data pengukuran valid.")

    df = df.sort_values("Datetime").reset_index(drop=True)

    print("Dataset berhasil dimuat.")
    print("Jumlah baris:", len(df))
    print("Periode:", df["Datetime"].min(), "hingga", df["Datetime"].max())
    print("Jumlah variabel pengukuran:", len(kolom_pengukuran))

except Exception as error:
    raise RuntimeError(
        "Dataset tidak berhasil dimuat. Periksa koneksi internet "
        "dan coba jalankan kembali cell ini. Tidak ada data dummy "
        "yang digunakan sebagai pengganti."
    ) from error


# ============================================================
# SESI 2 - USER-CENTERED DESIGN (UCD)
# ============================================================

display(Markdown("""
## Sesi 2 — User-Centered Design

Dasbor dirancang berdasarkan kebutuhan pengguna, bukan sekadar
tampilan visual. Pengguna harus dapat memilih variabel pengukuran,
rentang tanggal, dan melihat informasi yang relevan.

**Contoh kebutuhan pengguna:**
1. Mengetahui perubahan konsentrasi polutan dari waktu ke waktu.
2. Memilih variabel pengukuran yang ingin dianalisis.
3. Membatasi periode analisis.
4. Melihat nilai statistik dan data terperinci.
5. Mengunduh hasil analisis.

Tahap UCD yang diterapkan dalam prototipe ini adalah perancangan,
implementasi, evaluasi, dan perbaikan berdasarkan hasil pengujian.
"""))


# ============================================================
# SESI 3 - ASPEK KOGNITIF DAN PERSEPSI VISUAL
# ============================================================

display(Markdown("""
## Sesi 3 — Aspek Kognitif dan Persepsi Visual

Antarmuka menggunakan pengelompokan informasi agar pengguna
tidak harus memahami seluruh data sekaligus.

Penerapan prinsip desain:
- **Proximity:** filter dikelompokkan di bagian kontrol.
- **Similarity:** informasi statistik ditampilkan dalam format seragam.
- **Visual hierarchy:** judul, ringkasan, grafik, lalu tabel.
- **Kesederhanaan:** grafik hanya menampilkan variabel yang dipilih.
- **Keterbacaan:** label dan sumbu grafik diberi keterangan.
"""))


# ============================================================
# SESI 4 - AKSESIBILITAS DAN ETIKA DESAIN
# ============================================================

display(Markdown("""
## Sesi 4 — Aksesibilitas dan Etika Desain

Prinsip POUR diterapkan dalam rancangan:
- **Perceivable:** grafik dilengkapi judul, label sumbu, dan tabel.
- **Operable:** kontrol dapat digunakan melalui widget interaktif.
- **Understandable:** nama variabel dan pesan menggunakan bahasa jelas.
- **Robust:** menggunakan komponen Python dan Jupyter yang umum.

Etika dan transparansi:
- Sumber dataset dicantumkan.
- Nilai yang hilang tidak diganti dengan angka buatan.
- Filter tidak mengubah nilai asli secara diam-diam.
- Pengguna memilih sendiri variabel yang ingin dianalisis.
- Grafik bukan pengganti penilaian resmi kualitas udara.
"""))


# ============================================================
# SESI 5 - PROTOTIPE DAN EVALUASI USABILITY
# ============================================================

display(Markdown("""
## Sesi 5 — Prototipe dan Evaluasi Usability

Prototipe dibuat menggunakan Plotly dan ipywidgets.

Evaluasi usability dilakukan dengan mencatat:
- Keberhasilan menyelesaikan tugas.
- Waktu penyelesaian tugas.
- Kendala yang dialami pengguna.

Hasil pengujian harus berasal dari penggunaan nyata.
Kode tidak membuat nilai keberhasilan atau waktu pengujian palsu.
"""))


# ============================================================
# SESI 6 - ANALISIS KEBUTUHAN DAN USER PERSONA
# ============================================================

display(Markdown("""
## Sesi 6 — Analisis Kebutuhan dan User Persona

### Kebutuhan fungsional
- Memilih variabel pengukuran.
- Memilih periode analisis.
- Menampilkan statistik, grafik, dan tabel.
- Mengunduh data hasil filter.

### Kebutuhan nonfungsional
- Tampilan mudah dibaca.
- Filter memberikan respons yang jelas.
- Sumber data dan keterbatasannya terlihat.
- Kesalahan pemuatan data dilaporkan kepada pengguna.

### Persona ilustratif
**Pengguna:** mahasiswa yang mempelajari visualisasi data.

**Goals:** memahami tren pengukuran dan menyajikan hasil analisis.

**Pain points:** tabel besar sulit dibaca dan grafik tanpa filter
sulit digunakan untuk memeriksa periode tertentu.

**Behaviors:** memilih variabel, memeriksa grafik, lalu melihat
data terperinci.

Persona ini merupakan contoh untuk latihan desain, bukan hasil
wawancara atau survei pengguna sungguhan.
"""))


# ============================================================
# SESI 7 - DASBOR INTERAKTIF DAN VISUALISASI DATA
# ============================================================

display(Markdown("""
## Sesi 7 — Dasbor Interaktif Kualitas Udara

Dasbor menerapkan tiga prinsip Shneiderman:

1. **Overview First:** menampilkan ringkasan statistik.
2. **Zoom and Filter:** pengguna memilih variabel dan periode.
3. **Details-on-Demand:** pengguna dapat melihat tabel data.

Prinsip Edward Tufte juga diterapkan dengan menghindari dekorasi
grafik berlebihan agar perhatian pengguna terarah pada informasi.
"""))

# Pilih kolom pengukuran yang paling informatif sebagai nilai awal.
pilihan_awal = (
    "CO(GT)" if "CO(GT)" in kolom_pengukuran
    else kolom_pengukuran[0]
)

variabel = widgets.Dropdown(
    options=kolom_pengukuran,
    value=pilihan_awal,
    description="Variabel:",
    layout=widgets.Layout(width="450px"),
    style={"description_width": "100px"}
)

tanggal_awal = widgets.DatePicker(
    description="Tanggal awal:",
    value=df["Datetime"].min().date(),
    style={"description_width": "100px"}
)

tanggal_akhir = widgets.DatePicker(
    description="Tanggal akhir:",
    value=df["Datetime"].max().date(),
    style={"description_width": "100px"}
)

jumlah_baris = widgets.IntSlider(
    value=15,
    min=5,
    max=50,
    step=5,
    description="Baris tabel:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="450px")
)

tombol = widgets.Button(
    description="Tampilkan analisis",
    button_style="primary",
    icon="refresh"
)

tombol_unduh = widgets.Button(
    description="Siapkan CSV",
    button_style="success",
    icon="download"
)

keluaran = widgets.Output()
keluaran_csv = widgets.Output()

# Menyimpan hasil filter terbaru untuk diekspor.
hasil_terfilter = {"data": pd.DataFrame()}


def tampilkan_dashboard(_=None):
    with keluaran:
        clear_output(wait=True)

        if tanggal_awal.value is None or tanggal_akhir.value is None:
            print("Silakan pilih tanggal awal dan tanggal akhir.")
            return

        if tanggal_awal.value > tanggal_akhir.value:
            print("Tanggal awal tidak boleh melewati tanggal akhir.")
            return

        mulai = pd.Timestamp(tanggal_awal.value)
        selesai = pd.Timestamp(tanggal_akhir.value) + pd.Timedelta(days=1)

        data = df.loc[
            (df["Datetime"] >= mulai)
            & (df["Datetime"] < selesai),
            ["Datetime", variabel.value]
        ].dropna().copy()

        if data.empty:
            hasil_terfilter["data"] = pd.DataFrame()
            print(
                "Tidak ada pengukuran valid pada periode dan variabel "
                "yang dipilih. Silakan ubah filter."
            )
            return

        hasil_terfilter["data"] = data.copy()

        nilai = data[variabel.value]
        rata_rata = nilai.mean()
        median = nilai.median()
        minimum = nilai.min()
        maksimum = nilai.max()

        display(Markdown(
            f"### Ringkasan: {variabel.value}\n"
            f"Periode: **{mulai.date()} sampai "
            f"{tanggal_akhir.value}**"
        ))

        # Ringkasan statistik
        ringkasan = pd.DataFrame({
            "Indikator": [
                "Jumlah pengukuran valid",
                "Rata-rata",
                "Median",
                "Minimum",
                "Maksimum"
            ],
            "Nilai": [
                len(nilai),
                round(rata_rata, 3),
                round(median, 3),
                round(minimum, 3),
                round(maksimum, 3)
            ]
        })

        display(ringkasan)

        # Grafik tren berdasarkan waktu
        fig_tren = px.line(
            data,
            x="Datetime",
            y=variabel.value,
            title=f"Tren {variabel.value} Berdasarkan Waktu",
            labels={
                "Datetime": "Tanggal dan waktu",
                variabel.value: f"{variabel.value} (satuan dataset)"
            },
            template="plotly_white"
        )

        fig_tren.update_layout(
            height=430,
            hovermode="x unified",
            margin=dict(l=40, r=25, t=65, b=50)
        )

        display(fig_tren)

        # Distribusi nilai
        fig_distribusi = px.histogram(
            data,
            x=variabel.value,
            nbins=30,
            title=f"Distribusi Nilai {variabel.value}",
            labels={
                variabel.value: f"{variabel.value} (satuan dataset)",
                "count": "Frekuensi"
            },
            template="plotly_white"
        )

        fig_distribusi.update_layout(
            height=380,
            margin=dict(l=40, r=25, t=65, b=50)
        )

        display(fig_distribusi)

        # Tabel detail berdasarkan jumlah baris yang dipilih
        display(Markdown("### Detail pengukuran"))
        display(
            data.tail(jumlah_baris.value)
            .sort_values("Datetime", ascending=False)
            .reset_index(drop=True)
        )

        print(
            "Catatan: data yang kosong telah dikeluarkan dari "
            "perhitungan. Hasil hanya mewakili variabel dan periode "
            "yang dipilih, bukan indeks kualitas udara resmi."
        )


def siapkan_csv(_):
    with keluaran_csv:
        clear_output(wait=True)

        data = hasil_terfilter["data"]

        if data.empty:
            print("Tampilkan analisis terlebih dahulu sebelum ekspor.")
            return

        nama_file = "hasil_analisis_kualitas_udara.csv"

        data.to_csv(nama_file, index=False, encoding="utf-8-sig")

        from google.colab import files
        files.download(nama_file)


tombol.on_click(tampilkan_dashboard)
tombol_unduh.on_click(siapkan_csv)

display(Markdown("### Kontrol dasbor"))
display(variabel)
display(widgets.HBox([tanggal_awal, tanggal_akhir]))
display(jumlah_baris)
display(widgets.HBox([tombol, tombol_unduh]))
display(keluaran)
display(keluaran_csv)

# Menampilkan analisis pertama secara otomatis
tampilkan_dashboard()


# ============================================================
# EVALUASI USABILITY - SESI 5
# ============================================================

display(Markdown("""
---
## Praktik evaluasi usability

Gunakan dasbor di atas untuk menyelesaikan tugas berikut.
Isi hasil berdasarkan pengujian yang benar-benar dilakukan.

**Contoh tugas:**
1. Pilih satu variabel pengukuran.
2. Atur rentang tanggal.
3. Temukan nilai rata-rata pada ringkasan.
4. Periksa tren pada grafik.
5. Tampilkan tabel dan siapkan ekspor CSV.

Success rate dihitung sebagai jumlah tugas yang berhasil dibagi
jumlah seluruh tugas, dikalikan 100%.
"""))

jumlah_tugas = widgets.IntText(
    value=5,
    description="Total tugas:",
    style={"description_width": "initial"}
)

tugas_berhasil = widgets.IntText(
    value=0,
    description="Tugas berhasil:",
    style={"description_width": "initial"}
)

waktu_detik = widgets.FloatText(
    value=0.0,
    description="Waktu (detik):",
    style={"description_width": "initial"}
)

catatan_pengujian = widgets.Textarea(
    description="Kendala:",
    placeholder="Tuliskan kendala yang benar-benar dialami pengguna...",
    layout=widgets.Layout(width="95%", height="90px"),
    style={"description_width": "initial"}
)

tombol_evaluasi = widgets.Button(
    description="Hitung hasil evaluasi",
    button_style="info"
)

keluaran_evaluasi = widgets.Output()


def hitung_evaluasi(_):
    with keluaran_evaluasi:
        clear_output(wait=True)

        total = jumlah_tugas.value
        berhasil = tugas_berhasil.value
        waktu = waktu_detik.value

        if total <= 0:
            print("Total tugas harus lebih besar dari nol.")
            return

        if berhasil < 0 or berhasil > total:
            print("Jumlah tugas berhasil harus antara 0 dan total tugas.")
            return

        if waktu < 0:
            print("Waktu tidak boleh bernilai negatif.")
            return

        success_rate = berhasil / total * 100

        hasil = pd.DataFrame({
            "Metrik": [
                "Total tugas",
                "Tugas berhasil",
                "Success rate (%)",
                "Waktu pengujian (detik)",
                "Catatan kendala"
            ],
            "Hasil": [
                total,
                berhasil,
                round(success_rate, 2),
                waktu,
                catatan_pengujian.value or "Tidak diisi"
            ]
        })

        display(hasil)

        print(
            "Interpretasi: success rate menunjukkan proporsi tugas "
            "yang berhasil diselesaikan. Waktu dan kendala perlu "
            "dibandingkan antar pengguna atau sesi pengujian "
            "sebelum menyimpulkan kualitas usability."
        )


tombol_evaluasi.on_click(hitung_evaluasi)

display(widgets.VBox([
    jumlah_tugas,
    tugas_berhasil,
    waktu_detik,
    catatan_pengujian,
    tombol_evaluasi,
    keluaran_evaluasi
]))


# ============================================================
# PENUTUP DAN SUMBER DATA
# ============================================================

display(Markdown("""
---
## Kesimpulan implementasi

Prototipe ini menerapkan konsep HCI dan visualisasi data melalui
antarmuka yang memiliki kontrol filter, ringkasan statistik,
grafik tren, distribusi nilai, tabel detail, dan ekspor CSV.

Evaluasi usability disediakan agar pengujian dapat dilakukan
menggunakan hasil interaksi nyata. Kebutuhan aksesibilitas,
konsistensi, dan kesederhanaan juga diperhatikan dalam rancangan.

**Sumber dataset:**

UCI Machine Learning Repository — Air Quality Data Set

https://archive.ics.uci.edu/dataset/360/air+quality

**Catatan:** dataset berasal dari Italia dan mengandung pengukuran
sensor serta nilai yang hilang. Hasil analisis tidak boleh dianggap
sebagai pengukuran kualitas udara terkini atau kondisi Indonesia.
"""))



# STUDI KASUS: DASBOR INTERAKTIF KUALITAS UDARA

**Materi:** Human-Computer Interaction & Data Visualization

**Tujuan:** Mengembangkan prototipe dasbor yang membantu pengguna
memahami perubahan kualitas udara melalui grafik dan filter interaktif.

Dataset yang digunakan merupakan hasil pengukuran sensor kualitas
udara di Italia dari UCI Machine Learning Repository.

Prinsip yang diterapkan:
- Human-Computer Interaction (HCI)
- User-Centered Design (UCD)
- Visualisasi data
- Aksesibilitas dan usability
- Dasbor interaktif


Mengunduh dataset asli dari UCI...
Dataset berhasil dimuat.
Jumlah baris: 9357
Periode: 2004-03-10 18:00:00 hingga 2005-04-04 14:00:00
Jumlah variabel pengukuran: 13



## Sesi 2 — User-Centered Design

Dasbor dirancang berdasarkan kebutuhan pengguna, bukan sekadar
tampilan visual. Pengguna harus dapat memilih variabel pengukuran,
rentang tanggal, dan melihat informasi yang relevan.

**Contoh kebutuhan pengguna:**
1. Mengetahui perubahan konsentrasi polutan dari waktu ke waktu.
2. Memilih variabel pengukuran yang ingin dianalisis.
3. Membatasi periode analisis.
4. Melihat nilai statistik dan data terperinci.
5. Mengunduh hasil analisis.

Tahap UCD yang diterapkan dalam prototipe ini adalah perancangan,
implementasi, evaluasi, dan perbaikan berdasarkan hasil pengujian.



## Sesi 3 — Aspek Kognitif dan Persepsi Visual

Antarmuka menggunakan pengelompokan informasi agar pengguna
tidak harus memahami seluruh data sekaligus.

Penerapan prinsip desain:
- **Proximity:** filter dikelompokkan di bagian kontrol.
- **Similarity:** informasi statistik ditampilkan dalam format seragam.
- **Visual hierarchy:** judul, ringkasan, grafik, lalu tabel.
- **Kesederhanaan:** grafik hanya menampilkan variabel yang dipilih.
- **Keterbacaan:** label dan sumbu grafik diberi keterangan.



## Sesi 4 — Aksesibilitas dan Etika Desain

Prinsip POUR diterapkan dalam rancangan:
- **Perceivable:** grafik dilengkapi judul, label sumbu, dan tabel.
- **Operable:** kontrol dapat digunakan melalui widget interaktif.
- **Understandable:** nama variabel dan pesan menggunakan bahasa jelas.
- **Robust:** menggunakan komponen Python dan Jupyter yang umum.

Etika dan transparansi:
- Sumber dataset dicantumkan.
- Nilai yang hilang tidak diganti dengan angka buatan.
- Filter tidak mengubah nilai asli secara diam-diam.
- Pengguna memilih sendiri variabel yang ingin dianalisis.
- Grafik bukan pengganti penilaian resmi kualitas udara.


...

IntSlider(value=15, description='Baris tabel:', layout=Layout(width='450px'), max=50, min=5, step=5, style=Sli…

Output()

Output()


---
## Praktik evaluasi usability

Gunakan dasbor di atas untuk menyelesaikan tugas berikut.
Isi hasil berdasarkan pengujian yang benar-benar dilakukan.

**Contoh tugas:**
1. Pilih satu variabel pengukuran.
2. Atur rentang tanggal.
3. Temukan nilai rata-rata pada ringkasan.
4. Periksa tren pada grafik.
5. Tampilkan tabel dan siapkan ekspor CSV.

Success rate dihitung sebagai jumlah tugas yang berhasil dibagi
jumlah seluruh tugas, dikalikan 100%.



---
## Kesimpulan implementasi

Prototipe ini menerapkan konsep HCI dan visualisasi data melalui
antarmuka yang memiliki kontrol filter, ringkasan statistik,
grafik tren, distribusi nilai, tabel detail, dan ekspor CSV.

Evaluasi usability disediakan agar pengujian dapat dilakukan
menggunakan hasil interaksi nyata. Kebutuhan aksesibilitas,
konsistensi, dan kesederhanaan juga diperhatikan dalam rancangan.

**Sumber dataset:**

UCI Machine Learning Repository — Air Quality Data Set

https://archive.ics.uci.edu/dataset/360/air+quality

**Catatan:** dataset berasal dari Italia dan mengandung pengukuran
sensor serta nilai yang hilang. Hasil analisis tidak boleh dianggap
sebagai pengukuran kualitas udara terkini atau kondisi Indonesia.
